# Step 6: generate MBBQ model outputs on Colab

Runs `my_research/generation/run_step6.py` for one model over en, nl, es, tr x {main, control}.
Everything is written to Google Drive, and finished subsets are skipped, so **after a disconnect just run all cells again**.

Before the first run:
1. Runtime > Change runtime type > **A100** (L4 also works). A T4 lacks bf16 and the script refuses it.
2. Optional: add a Colab secret (key icon on the left) named `HF_TOKEN` holding a read token, with notebook access on. Neither model is gated; a token only avoids anonymous rate limits.
3. Push `my_research/` to the fork (cell 2 clones it from GitHub).

In [ ]:
# Which model: run once with "mistral", then once with "zephyr".
MODEL_KEY = "mistral"
HF_MODEL = {"mistral": "mistralai/Mistral-7B-Instruct-v0.2",
            "zephyr": "HuggingFaceH4/zephyr-7b-beta"}[MODEL_KEY]
REVISION = ""  # empty = current main (recorded); set a commit hash to pin
REV_ARG = f"--revision {REVISION}" if REVISION else ""
BS = 16        # batch size used for the reproduction; keep it fixed
DRIVE = "/content/drive/MyDrive/mbbq"
REPO = "https://github.com/michaellarionov/MBBQ_Russian.git"
TRANSFORMERS_PIN = "4.39.3"  # requirements.txt version; "" = Colab's own

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
try:  # optional; mbbq.py's login is skipped and the hub reads HF_TOKEN
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    pass
if not os.path.isdir("/content/MBBQ_Russian"):
    !git clone -q {REPO} /content/MBBQ_Russian
%cd /content/MBBQ_Russian
!git pull -q && git log --oneline -1
if TRANSFORMERS_PIN:
    !pip install -q transformers=={TRANSFORMERS_PIN}
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.get_device_name())"

## Smoke test (do this first)

One subset (Gender_identity, English, main set: 2,720 prompts). It checks that loading, generation and answer detection work, and it estimates the GPU time for the full run.

In [ ]:
!python my_research/generation/run_step6.py --key {MODEL_KEY}_smoke --hf_model {HF_MODEL} \
    {REV_ARG} --bs {BS} \
    --langs en --subsets Gender_identity --no_control \
    --work_dir {DRIVE}/work --results_dir {DRIVE}/smoke

In [ ]:
import json, sys
sys.path.insert(0, "my_research/scorer")
from mbbq_scorer import annotate, attach_polarity, load_outputs, score

smoke = load_outputs(f"{DRIVE}/smoke/{MODEL_KEY}_smoke_en.pkl")
print(smoke[["prompt_id", "answer", "answer_detected"]].sample(5, random_state=0).to_string())
s = score(annotate(attach_polarity(smoke, "data", "en")))
print({k: round(v, 3) for k, v in s.items()})

info = json.load(open(f"{DRIVE}/smoke/{MODEL_KEY}_smoke_runinfo.json"))
secs = sum(info["ask_seconds"].values())
full = 8 * 50_400  # prompts: 4 languages x {main, control}
print(f"{info['gpu']}: {secs / len(smoke):.3f} s/prompt -> "
      f"~{secs / len(smoke) * full / 3600:.0f} GPU-hours for the full run")

Continue only if the answers look sensible, few are undetected, and the GPU-hour estimate fits your compute units.

## Full run

This takes many hours. If Colab disconnects, run all cells again, skipping the smoke-test cells; it resumes at the unfinished subset.

In [ ]:
!python my_research/generation/run_step6.py --key {MODEL_KEY} --hf_model {HF_MODEL} \
    {REV_ARG} --bs {BS} \
    --work_dir {DRIVE}/work --results_dir {DRIVE}/results

In [ ]:
info = json.load(open(f"{DRIVE}/results/{MODEL_KEY}_runinfo.json"))
print(json.dumps({k: v for k, v in info.items() if k != "ask_seconds"}, indent=1))
print(f"GPU time so far: {sum(info['ask_seconds'].values()) / 3600:.1f} h")

When all 8 `<model>_<lang>[_control].pkl` files and `<model>_runinfo.json` are in `MyDrive/mbbq/results`, download them into `my_research/results/` locally (about 50 MB per pickle) and run step 7. Copy the revision hash from the runinfo into `my_research/README.md`.